In [43]:
import pandas as pd
from pathlib import Path
data_dir = Path("./mock_dataset")
dataset = {}

for f in data_dir.glob("*.csv"):
    df = pd.read_csv(f)
    dataset[f.stem] = df

In [44]:
dataset.keys()

dict_keys(['mock_life_insurance_products', 'sales_script_by_stage', 'trigger_detection_dataset_template', 'trigger_label_schema', 'trigger_response_matrix'])

In [3]:
print(dataset['trigger_response_matrix'].head().to_string())

                      trigger_phrase intent_category                     detection_keywords                                                                                                                                                                                  recommended_response                                                          ai_actions                                                     personalization_tokens                                              next_best_step  priority                                         notes
0             How much does it cost?        Interest            cost;price;premium;how much  Our plans start from {currency}{min_premium}/month and can be tailored to your budget. Based on {persona}, a coverage of {currency}{suggested_coverage} often balances affordability and protection.           surface_affordable_plans;show_quote_range;log_intent_cost                    {currency},{min_premium},{persona},{suggested_coverage}  Confirm budget ra

In [4]:
intent_message = dataset['trigger_response_matrix'].groupby("intent_category").agg(message=pd.NamedAgg(column="trigger_phrase", aggfunc=lambda x: list(set(x)))).reset_index()
intent_message

,intent_category,message
0,Hesitation,"[I already have some coverage., I don't trust ..."
1,Information,"[Is there a waiting period?, What happens if I..."
2,Interest,"[How much does it cost?, Is there a cash value..."
3,Risk,"[What if the company goes bankrupt?, What if I..."
4,Urgency,"[Can you send me the application now?, Can we ..."


In [5]:
print("TRIGGER MESSAGE")
for im in intent_message.to_dict(orient="records"):
    prompt = "{intent}:\n{message}".format(intent=im['intent_category'], message="\n".join([f"\t- {i}" for i in im['message']]))
    print(prompt)

TRIGGER MESSAGE
Hesitation:
	- I already have some coverage.
	- I don't trust insurance companies.
	- I'll talk to my spouse first.
	- I hate paperwork.
	- I'm healthy, so I don't think I need it.
	- I'm not sure if I need this.
	- I'm comparing other options.
	- Not interested.
	- I've had a bad experience before.
	- I'll wait until next year.
	- I'm worried about the cost.
	- I fear hidden fees.
	- I'm too young for life insurance.
	- I need to think about it.
	- I don't have time right now.
Information:
	- Is there a waiting period?
	- What happens if I move abroad?
	- What's the suicide clause?
	- What happens if I cancel?
	- Can I name multiple beneficiaries?
	- Is this tax-deductible?
	- What's the difference between term and whole life?
	- What documents do I need?
	- How long does the process take?
	- How soon does coverage start?
	- Can I reinstate a lapsed policy?
	- What's the claim process like?
	- Will premiums increase over time?
	- Can I take a policy loan?
	- Are there 

In [6]:
# Simple keyword matching for speed
def detect_intent_keywords(message, response_df):
    message = message.lower()
    for _, row in response_df.iterrows():
        keywords = row['detection_keywords'].split(';')
        if any(kw.strip().lower() in message for kw in keywords):
            return {
                'intent': row['intent_category'],
                'trigger': row['trigger_phrase'],
                'response': row['recommended_response'],
                'actions': row['ai_actions'],
                'priority': row['priority']
            }
    return None

# Test with your data
response_df = dataset['trigger_response_matrix']

# Examples
test_messages = [
    "How expensive is this plan?",  # Should match "cost;price;premium"
    "Can we start the application today?",  # Should match "start today"
    "Do you have plans for kids?"  # Should match "child;children"
]

for msg in test_messages:
    result = detect_intent_keywords(msg, response_df)
    if result:
        print(f"Message: '{msg}'")
        print(f"Intent: {result['intent']}")
        print(f"Priority: {result['priority']}")
        print(f"Actions: {result['actions']}")
        print("---")

Message: 'How expensive is this plan?'
Intent: Hesitation
Priority: 1
Actions: budget_slider;recommend_affordable_plan;log_objection_cost
---
Message: 'Can we start the application today?'
Intent: Information
Priority: 3
Actions: link_to_portal;log_info_online
---


In [7]:
def detect_intent_realtime(message, response_df):
    message = message.lower()
    for _, row in response_df.iterrows():
        keywords = row['detection_keywords'].split(';')
        if any(kw.strip().lower() in message for kw in keywords):
            return {
                'intent': row['intent_category'],
                'response': row['recommended_response'],
                'actions': row['ai_actions'].split(';'),
                'priority': row['priority']
            }
    return None

# Usage for live calls
response_df = dataset['trigger_response_matrix']

def process_live_transcript(transcript_chunk):
    result = detect_intent_realtime(transcript_chunk, response_df)
    if result:
        print(f"🎯 Intent: {result['intent']} (Priority: {result['priority']})")
        print(f"💬 Suggested response: {result['response'][:100]}...")
        print(f"⚡ Actions: {result['actions'][:3]}")  # Show first 3 actions
        return result
    return None


In [8]:
_ = process_live_transcript("How expensive is this plan?")

🎯 Intent: Hesitation (Priority: 1)
💬 Suggested response: Let’s set a comfortable budget first, then tailor coverage. Most clients choose {currency}{premium_t...
⚡ Actions: ['budget_slider', 'recommend_affordable_plan', 'log_objection_cost']


In [9]:
dataset['mock_life_insurance_products'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28 entries, 0 to 27
Data columns (total 24 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  28 non-null     object 
 1   product_name                28 non-null     object 
 2   product_type                28 non-null     object 
 3   objective                   28 non-null     object 
 4   age_min                     28 non-null     int64  
 5   age_max                     28 non-null     int64  
 6   requires_medical_exam       28 non-null     bool   
 7   medical_exam_threshold_thb  18 non-null     float64
 8   health_requirements         25 non-null     object 
 9   coverage_min_thb            28 non-null     int64  
 10  coverage_max_thb            28 non-null     int64  
 11  premium_min_month_thb       28 non-null     int64  
 12  premium_max_month_thb       28 non-null     int64  
 13  payment_frequency_options   28 non-nu

In [10]:
from toon import encode, decode

test_obj = dict(data=[{"role": "user", "content": "Hello, how are you?"}])
toon_str = encode(test_obj)
print(toon_str)

data[1,]{role,content}:
  user,"Hello, how are you?"


In [11]:
decode(toon_str)

{'data': [{'role': 'user', 'content': 'Hello, how are you?'}]}

In [18]:
from backend.llms.bedrock import BedrockNova
from backend.llms.base import UserMessage, AIMessage
from backend.llms.utils import parse_blockcode
llm = BedrockNova("us.amazon.nova-micro-v1:0")

In [15]:
structured_output = encode(dict(sentiment="posivive|neutral|negative"))

In [16]:
system_prompt = """
# PERSONA
You are the best mind reader. Your job is to classify user sentiment into: positive, neutral, negative.

# INSTRUCTION
- read USER_INPUT and classify user's sentiment into: positive, neutral, negative

# OUTPUT
Reply only in the format below:
```toon
{structured_output}
```
""".format(structured_output=structured_output)

messages = [UserMessage(content="I hate this pizza")]
response = llm.run(system_prompt=system_prompt, messages=messages)

In [17]:
print(response.content)

```toon
sentiment: negative
```


In [19]:
toon_str = parse_blockcode(response.content, "toon")
decode(toon_str)

{'sentiment': 'negative'}

In [56]:
structured_output = encode(dict(age="a positive number default null", salary="a positive number default null"))

system_prompt = """
# PERSONA
You are the best information extractor. Your job is to extract customer information

# INSTRUCTION
- read TEXT and extract information

# OUTPUT
Reply only in the format below:
```toon
{structured_output}
```
""".format(structured_output=structured_output)

content = """\
TEXT:
I'm twenty-two.
Well then you couldn't have this one, but you can have this one. 
Also most customers love this product A because bla bla bla and the price is 150 USD. per month. 
I think it above my budget because now I make only three thousands USD per month.
"""

messages = [UserMessage(content=content)]
response = llm.run(system_prompt=system_prompt, messages=messages)

In [57]:
customer_info = decode(parse_blockcode(response.content, "toon"))
customer_info

{'age': 22, 'salary': 3000}

In [58]:
product_df = dataset['mock_life_insurance_products']
product_df.head()

,product_id,product_name,product_type,objective,age_min,age_max,requires_medical_exam,medical_exam_threshold_thb,health_requirements,coverage_min_thb,...,term_years,cash_value,indexation_available,underwriting_level,riders_available,target_segments,promo_eligibility,currency,regulator,notes
0,ST20,SecureTerm 20,Term,Pure protection,18,60,True,8000000.0,Basic health questionnaire,1000000,...,20.0,False,True,Standard,Accidental Death;Critical Illness;Hospital Income,Young professionals;Families,Annual payment 5% off,THB,OIC,Competitive pricing for ages 25–45
1,ST30,SecureTerm 30,Term,Long-term protection,18,55,True,6000000.0,Basic health questionnaire,1000000,...,30.0,False,True,Standard,Accidental Death;Critical Illness,Mortgage holders;New parents,Bundle with AccidentGuard 10% off,THB,OIC,Designed for long liabilities like mortgages
2,LPWL,LegacyPlus Whole,Whole Life,Legacy + savings,25,65,True,5000000.0,Medical exam over threshold,2000000,...,NaN,True,True,Full,Critical Illness;Waiver of Premium,High earners;Business owners,Annual payment 7% off,THB,OIC,Cash value accumulation with dividend potential
3,WLCS,WealthLife Cash Saver,Whole Life,Savings focus,25,60,True,4000000.0,Medical exam over threshold,1000000,...,NaN,True,True,Full,Waiver of Premium;Accidental Death,Savers;Parents,Early-bird promo Q1,THB,OIC,High cash value build-up from year 3
4,EGCH,EduShield Child,Child/Term,Education protection,20,55,False,NaN,No major illness in parent,500000,...,18.0,False,True,Standard,Waiver of Premium;Parent AD&D,Parents with young children,School season promo,THB,OIC,Premium waiver if parent dies or disabled


In [63]:
def product_filter_by_customer_info(dataset=None, age=None, salary=None):
    mask = True
    if age:
        mask &= dataset['age_min'] <= age
        mask &= dataset['age_max'] >= age
    if salary:
        afford = salary * 0.1
        mask &= dataset['premium_min_month_thb'] <= afford
        mask &= dataset['premium_max_month_thb'] >= afford
    # print(mask)
    return dataset.loc[mask,:]

In [64]:
product_filter_by_customer_info(dataset=product_df, **customer_info)

,product_id,product_name,product_type,objective,age_min,age_max,requires_medical_exam,medical_exam_threshold_thb,health_requirements,coverage_min_thb,...,term_years,cash_value,indexation_available,underwriting_level,riders_available,target_segments,promo_eligibility,currency,regulator,notes
7,ACC1,AccidentGuard Basic,Accident,Accident coverage,18,70,False,NaN,NaN,1000000,...,1.0,False,False,NaN,—,Gig workers;Drivers,No medical discount,THB,OIC,"Fast issue, no underwriting"
17,MICRO,MicroTerm Starter,Term,Starter protection,18,35,False,NaN,No exam; declaration only,300000,...,10.0,False,True,Simplified,Accidental Death,Students;First-jobbers,Digital-only 3% off,THB,OIC,"Low cost entry, mobile onboarding"
20,TRV1,TravelAccident Global,Accident/Travel,Travel accident,18,65,False,NaN,NaN,1000000,...,1.0,False,False,NaN,—,Frequent travelers,Holiday promo,THB,OIC,Worldwide accident cover (non-medical)
23,UNI1,UniStudent Protect,Term,Student budget protection,18,28,False,NaN,Declaration only,200000,...,5.0,False,True,Simplified,Accidental Death,University students,Back-to-school promo,THB,OIC,Ultra-low premium for students


In [65]:
product_df.isna().sum()

product_id                     0
product_name                   0
product_type                   0
objective                      0
age_min                        0
age_max                        0
requires_medical_exam          0
medical_exam_threshold_thb    10
health_requirements            3
coverage_min_thb               0
coverage_max_thb               0
premium_min_month_thb          0
premium_max_month_thb          0
payment_frequency_options      0
term_years                     8
cash_value                     0
indexation_available           0
underwriting_level             4
riders_available               0
target_segments                0
promo_eligibility              0
currency                       0
regulator                      0
notes                          0
dtype: int64

In [1]:
from backend.llms.ollama import LocalEmbedding

lem = LocalEmbedding(model_name="bge-m3")

In [21]:
vectors = lem.run(texts=["I love this pizza", "Sushi is great"])

In [22]:
vector1 = vectors["embeddings"][0]
vector2 = vectors["embeddings"][1]

In [23]:
# from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# For 2D arrays (multiple vectors)
# similarity = cosine_similarity(vector1.reshape(1, -1), vector2.reshape(1, -1))[0][0]

# Or using numpy directly
def cosine_sim(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

cosine_sim(vector1, vector2)

np.float64(0.6500237417711925)